# Activation magnitudes of lost vs. generally active latents (App. F.4)

Reads the raw amplitudes written by `amplitude_lost_vs_general.py`
(`amplitude_raw_values__pooling=last_token.npz`, keys `layer={L}__k={k}__{general|lost}`):
*general* = activations (on t0) of all latents active on the noun of t0,
*lost* = the subset of those that are inactive on the noun of t_k.

Figures (rows = models, columns = SAE variants, as in `experiment_modeb_lostrate_by_k.png`):
1. `amplitude_hist__layerQ{1-4}__k{K}.png`: one grid per interpretability layer, log-x density histograms.
2. `amplitude_percentile_density__k{K}.png`: all four interpretability layers in one panel, scale-free:
   density of the percentile rank of each lost activation within the general distribution of the same
   layer (uniform = 1 if lost latents were a random subset). Inset: mean percentile rank over all layers.
3. `amplitude_mean_percentile_by_k.png`: mean percentile rank of lost activations vs. step k.
4. `amplitude_summary_stats.csv`: overlap coefficient and quartile shares quoted in App. F.4.

The paper figure (App. F.4) is `amplitude_hist__layerQ3__k1.png` (~65% depth, k = 1).

In [ ]:
from __future__ import annotations

import re
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import yaml
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

RESULTS_ROOT = Path("../../results/experiment_2_lost_latents")   # base_save_dir of amplitude_lost_vs_general.py
SAE_CONFIG_ROOT = Path("../../configs/sae")
OUTPUT_DIR = Path("wandb_model_grids_amplitude")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CASE = "english"
K_HIST = 1                 # step k used for the histogram / percentile-density figures
K_VALUES = [1, 2, 3, 4, 5]
N_HIST_BINS = 40
N_RANK_BINS = 10

MODEL_ORDER = ["gemma-3-270m", "gemma-3-1b-pt", "gemma-3-4b-pt", "gemma-3-12b-pt", "gemma-3-27b-pt",
               "gpt2-small", "meta-llama/Llama-3.1-8B", "mistralai/Mistral-7B-v0.1"]
PRETTY_MODEL = {"gemma-3-270m": "Gemma 3 270M", "gemma-3-1b-pt": "Gemma 3 1B", "gemma-3-4b-pt": "Gemma 3 4B",
                "gemma-3-12b-pt": "Gemma 3 12B", "gemma-3-27b-pt": "Gemma 3 27B", "gpt2-small": "GPT-2 Small",
                "meta-llama/Llama-3.1-8B": "Llama 3.1 8B", "mistralai/Mistral-7B-v0.1": "Mistral 7B"}
SAE_CONFIG_DIR = {"gemma-3-270m": "gemma-3-270m", "gemma-3-1b-pt": "gemma-3-1b", "gemma-3-4b-pt": "gemma-3-4b",
                  "gemma-3-12b-pt": "gemma-3-12b", "gemma-3-27b-pt": "gemma-3-27b", "gpt2-small": "gpt2-small",
                  "meta-llama/Llama-3.1-8B": "llama-3_1-8b", "mistralai/Mistral-7B-v0.1": "mistral-7b"}
SAE_ORDER = ["16k big", "16k medium", "16k small", "262k big", "262k medium", "262k small",
             "32k", "65k", "128k"]
DEPTH_LABELS = ["~25%", "~50%", "~65%", "~85%"]  # nominal targets of sae.interpretability_layers
N_LAYERS = {"gemma-3-270m": 18, "gemma-3-1b-pt": 26, "gemma-3-4b-pt": 34, "gemma-3-12b-pt": 48, "gemma-3-27b-pt": 62,
            "gpt2-small": 12, "meta-llama/Llama-3.1-8B": 32, "mistralai/Mistral-7B-v0.1": 32}


def depth(model: str, layer: int) -> str:
    return f"{round(100 * layer / N_LAYERS[model])}%"
LAYER_COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]
COLOR_GENERAL = "0.55"
COLOR_LOST = "crimson"

LABEL_FONTSIZE, TICK_FONTSIZE, TITLE_FONTSIZE, LEGEND_FONTSIZE, ROWLABEL_FONTSIZE = 10, 8, 10, 8, 12

## Locate runs and interpretability layers

In [ ]:
def sae_label_from_path(model: str, rel: Path) -> str:
    s = str(rel).lower()
    m = re.search(r"width_(\d+k)_l0_(big|medium|small)", s)
    if m:
        return f"{m.group(1)} {m.group(2)}"
    if "gpt2" in model:
        return re.search(r"v5-(\d+k)", s).group(1)
    if "llama" in model.lower():  # Llama Scope: 8x = 32k, 32x = 128k latents (d_model = 4096)
        return {"8x": "32k", "32x": "128k"}[re.search(r"lxr_(\d+x)", s).group(1)]
    if "mistral" in model.lower():
        return "65k"
    raise ValueError(f"unknown SAE path {rel}")


def model_from_path(rel: Path) -> str:
    parts = rel.parts
    return f"{parts[0]}/{parts[1]}" if parts[0] in ("meta-llama", "mistralai") else parts[0]


def interpretability_layers(model: str) -> List[int]:
    cfg_dir = SAE_CONFIG_ROOT / SAE_CONFIG_DIR[model]
    for f in [cfg_dir / "base.yaml", *sorted(cfg_dir.glob("*.yaml"))]:
        if f.exists():
            layers = (yaml.safe_load(f.read_text()) or {}).get("interpretability_layers")
            if layers:
                return [int(x) for x in layers]
    raise ValueError(f"no interpretability_layers for {model}")


runs: Dict[str, Dict[str, Path]] = defaultdict(dict)
for npz in sorted(RESULTS_ROOT.rglob(f"{CASE}/pooling=last_token/amplitude_plots/amplitude_raw_values__pooling=last_token.npz")):
    rel = npz.relative_to(RESULTS_ROOT)
    model = model_from_path(rel)
    if model not in MODEL_ORDER:
        continue
    runs[model][sae_label_from_path(model, rel)] = npz

models = [m for m in MODEL_ORDER if m in runs]
INTERP = {m: interpretability_layers(m) for m in models}
for m in models:
    print(f"{PRETTY_MODEL[m]:14s} layers {INTERP[m]}  SAEs {sorted(runs[m], key=SAE_ORDER.index)}")

## Helpers

In [ ]:
_cache: Dict[Path, dict] = {}


def load(npz: Path) -> dict:
    if npz not in _cache:
        with np.load(npz) as d:
            _cache[npz] = {k: d[k] for k in d.files}
    return _cache[npz]


def values(npz: Path, layer: int, k: int) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    d = load(npz)
    g, l = d.get(f"layer={layer}__k={k}__general"), d.get(f"layer={layer}__k={k}__lost")
    if g is None or l is None or g.size == 0:
        return None, None
    return g[g > 0], l[l > 0]


def available_layers(npz: Path, k: int) -> List[int]:
    return sorted(int(m.group(1)) for key in load(npz)
                  if (m := re.fullmatch(rf"layer=(\d+)__k={k}__general", key)))


def percentile_ranks(general: np.ndarray, lost: np.ndarray) -> np.ndarray:
    """Mid-rank percentile of each lost activation within the general distribution of the same layer."""
    g = np.sort(general)
    lo, hi = np.searchsorted(g, lost, "left"), np.searchsorted(g, lost, "right")
    return (lo + hi) / (2.0 * g.size)


def grid(models_: List[str]):
    n_cols = max(len(runs[m]) for m in models_)
    fig, axes = plt.subplots(len(models_), n_cols, figsize=(3.0 * n_cols, 2.55 * len(models_)), squeeze=False)
    return fig, axes, n_cols


def panels(models_: List[str], axes, n_cols: int):
    """Yield (ax, model, sae_label, npz, row, col); unused axes are switched off."""
    for r, m in enumerate(models_):
        labels = sorted(runs[m], key=SAE_ORDER.index)
        for c in range(n_cols):
            ax = axes[r, c]
            if c >= len(labels):
                ax.axis("off")
                continue
            yield ax, m, labels[c], runs[m][labels[c]], r, c
        axes[r, 0].annotate(PRETTY_MODEL[m], xy=(0, 0.5), xycoords="axes fraction", xytext=(-0.36, 0.5),
                            textcoords="axes fraction", rotation=90, ha="center", va="center",
                            fontsize=ROWLABEL_FONTSIZE, fontweight="bold")


def finish(fig, path: Path, suptitle: str):
    fig.suptitle(suptitle, fontsize=14, y=1.0)
    fig.tight_layout(rect=(0.02, 0, 1, 0.995))
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print("wrote", path)

## 1. One grid per interpretability layer: log-x histograms (general vs. lost)

In [ ]:
for q in range(4):
    fig, axes, n_cols = grid(models)
    for ax, m, sae, npz, r, c in panels(models, axes, n_cols):
        layer = INTERP[m][q] if q < len(INTERP[m]) else None
        g, l = values(npz, layer, K_HIST) if layer is not None else (None, None)
        if g is None:
            ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(sae, fontsize=TITLE_FONTSIZE)
            continue
        lo, hi = np.percentile(np.concatenate([g, l]), [0.5, 99.5])
        bins = np.logspace(np.log10(lo), np.log10(hi), N_HIST_BINS + 1)
        ax.hist(g, bins=bins, density=True, color=COLOR_GENERAL, alpha=0.45, label="active on $t_0$")
        ax.hist(l, bins=bins, density=True, histtype="step", color=COLOR_LOST, linewidth=1.6, label=f"lost at $t_{K_HIST}$")
        ax.axvline(np.median(g), color=COLOR_GENERAL, linestyle="--", linewidth=1.0)
        ax.axvline(np.median(l), color=COLOR_LOST, linestyle="--", linewidth=1.0)
        ax.set_xscale("log")
        ax.set_yticks([])
        ax.set_title(f"{sae} · layer {layer} ({depth(m, layer)})", fontsize=TITLE_FONTSIZE)
        ax.tick_params(labelsize=TICK_FONTSIZE)
        ax.text(0.98, 0.95, f"n={l.size:,} / {g.size:,}\nlost fraction {l.size / g.size:.2f}", transform=ax.transAxes,
                ha="right", va="top", fontsize=7, color="0.25")
        if c == 0:
            ax.set_ylabel("density", fontsize=LABEL_FONTSIZE)
        ax.set_xlabel("activation magnitude", fontsize=LABEL_FONTSIZE - 1)
        if r == 0 and c == 0:
            ax.legend(fontsize=LEGEND_FONTSIZE, loc="upper left", framealpha=0.9)
    finish(fig, OUTPUT_DIR / f"amplitude_hist__layerQ{q + 1}__k{K_HIST}.png",
           f"Activation magnitudes at the {DEPTH_LABELS[q]}-depth interpretability layer ({CASE}, k={K_HIST}); dashed: medians")

## 2. All four interpretability layers in one panel: percentile rank of lost activations

In [ ]:
fig, axes, n_cols = grid(models)
rank_edges = np.linspace(0, 1, N_RANK_BINS + 1)
centers = 0.5 * (rank_edges[:-1] + rank_edges[1:])
for ax, m, sae, npz, r, c in panels(models, axes, n_cols):
    for i, layer in enumerate(INTERP[m]):
        g, l = values(npz, layer, K_HIST)
        if g is None or l.size == 0:
            continue
        dens, _ = np.histogram(percentile_ranks(g, l), bins=rank_edges, density=True)
        ax.plot(centers, dens, marker="o", markersize=3, linewidth=1.8, color=LAYER_COLORS[i],
                label=f"layer {layer} ({depth(m, layer)})")
    ax.axhline(1.0, color="black", linestyle="--", linewidth=0.9)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 3.5)
    ax.set_title(sae, fontsize=TITLE_FONTSIZE)
    ax.tick_params(labelsize=TICK_FONTSIZE)
    ax.grid(True, linewidth=0.5, alpha=0.25)
    if c == 0:
        ax.set_ylabel("density of lost", fontsize=LABEL_FONTSIZE)
    ax.set_xlabel("percentile among active latents", fontsize=LABEL_FONTSIZE - 1)
    if c == 0:
        ax.legend(fontsize=LEGEND_FONTSIZE - 1, loc="lower left", framealpha=0.9)
    # inset: mean percentile rank of lost activations across all layers (0.5 = random subset)
    all_layers = available_layers(npz, K_HIST)
    mr = []
    for layer in all_layers:
        g, l = values(npz, layer, K_HIST)
        mr.append(np.mean(percentile_ranks(g, l)) if g is not None and l.size else np.nan)
    axin = inset_axes(ax, width="38%", height="32%", loc="upper right", borderpad=0.9)
    axin.plot(all_layers, mr, marker="o", markersize=2, linewidth=1.2, color="0.2")
    for i, layer in enumerate(INTERP[m]):
        if layer in all_layers:
            axin.plot(layer, mr[all_layers.index(layer)], "o", markersize=4, color=LAYER_COLORS[i])
    axin.axhline(0.5, color="black", linestyle="--", linewidth=0.7)
    axin.set_ylim(0.2, 0.6)
    axin.set_yticks([0.2, 0.4, 0.6])
    axin.tick_params(labelsize=6, pad=1)
    axin.patch.set_alpha(0.8)
finish(fig, OUTPUT_DIR / f"amplitude_percentile_density__k{K_HIST}.png",
       f"Where lost latents sit in the activation distribution ({CASE}, k={K_HIST}); dashed = random subset. "
       f"Inset: mean percentile vs. layer")

## 3. Mean percentile rank of lost activations vs. step k (format of the stepwise lost-rate figure)

In [ ]:
fig, axes, n_cols = grid(models)
for ax, m, sae, npz, r, c in panels(models, axes, n_cols):
    for i, layer in enumerate(INTERP[m]):
        ys = []
        for k in K_VALUES:
            g, l = values(npz, layer, k)
            ys.append(np.mean(percentile_ranks(g, l)) if g is not None and l.size else np.nan)
        ax.plot(K_VALUES, ys, linewidth=2.0, marker="o", markersize=3, color=LAYER_COLORS[i],
                label=f"layer {layer} ({depth(m, layer)})")
    ax.axhline(0.5, color="black", linestyle="--", linewidth=0.9)
    ax.set_ylim(0, 0.7)
    ax.set_xticks(K_VALUES)
    ax.set_title(sae, fontsize=TITLE_FONTSIZE)
    ax.tick_params(labelsize=TICK_FONTSIZE)
    ax.grid(True, linewidth=0.5, alpha=0.25)
    ax.set_xlabel("step $k$", fontsize=LABEL_FONTSIZE)
    if c == 0:
        ax.set_ylabel("mean percentile of lost", fontsize=LABEL_FONTSIZE)
        ax.legend(fontsize=LEGEND_FONTSIZE - 1, loc="lower right", framealpha=0.9)
finish(fig, OUTPUT_DIR / "amplitude_mean_percentile_by_k.png",
       f"Mean percentile of lost activations among active latents ({CASE}); dashed = random subset (0.5)")

## 4. Summary statistics quoted in App. F.4

Per SAE and interpretability layer (English, steps k = 1..5):
* **overlap**: overlap coefficient $\sum_b \min(p_b, q_b)$ between the magnitude distributions of lost and
  surviving latents (surviving = active minus lost), on the same 40 log-spaced bins as the histograms but with the
  range set by the 0.5/99.5 percentiles of the active distribution;
* **bottom / top quartile**: share of lost latents whose percentile rank among all active latents is
  < 0.25 / > 0.75 (0.25 each for a random subset).

The paper shows `amplitude_hist__layerQ3__k1.png` (~65% depth, k = 1) and quotes medians over SAEs (with ranges)
for that layer, for the other depths, and for larger k. All rows are written to `amplitude_summary_stats.csv`.

In [ ]:
import pandas as pd


def summary_stats(g: np.ndarray, l: np.ndarray) -> Tuple[float, float, float]:
    """(overlap coefficient lost vs. surviving, share of lost in bottom quartile, share in top quartile)."""
    bins = np.logspace(np.log10(np.percentile(g, 0.5)), np.log10(np.percentile(g, 99.5)), N_HIST_BINS + 1)
    h_gen, _ = np.histogram(g, bins)
    h_lost, _ = np.histogram(l, bins)
    h_surv = np.clip(h_gen - h_lost, 0, None)  # lost is a subset of active
    overlap = np.minimum(h_lost / h_lost.sum(), h_surv / h_surv.sum()).sum()
    pr = percentile_ranks(g, l)
    return overlap, (pr < 0.25).mean(), (pr > 0.75).mean()


rows = []
for m in models:
    for sae, npz in runs[m].items():
        for q, layer in enumerate(INTERP[m]):
            for k in K_VALUES:
                g, l = values(npz, layer, k)
                if g is None or l.size == 0:
                    continue
                ovl, bottom, top = summary_stats(g, l)
                rows.append(dict(model=m, sae=sae, depth=DEPTH_LABELS[q], layer=layer, k=k,
                                 overlap=ovl, bottom_quartile=bottom, top_quartile=top))
stats = pd.DataFrame(rows)
COLS = ["overlap", "bottom_quartile", "top_quartile"]
stats.to_csv(OUTPUT_DIR / "amplitude_summary_stats.csv", index=False)


def describe(df: pd.DataFrame) -> pd.Series:
    out = {"n": len(df)}
    for col in COLS:
        out[col] = f"{df[col].median():.2f} [{df[col].min():.2f}-{df[col].max():.2f}]"
    return pd.Series(out)


print("k = 1, per interpretability layer (median [range] over SAEs):")
display(stats[stats.k == 1].groupby("depth")[COLS].apply(describe))
print("~65% depth (shown in the paper), per k:")
display(stats[stats.depth == "~65%"].groupby("k")[COLS].apply(describe))
print("all four depths pooled, per k:")
display(stats.groupby("k")[COLS].apply(describe))